![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `meta-llama/llama-3-3-70b-instruct` to create AI service

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook provides a detailed demonstration of the steps and code required to showcase support for watsonx.ai AI service.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

This notebook aims to demonstrate the application of Chat models, such as `meta-llama/llama-3-3-70b-instruct`, using the tools provided by LangGraph. LangGraph serves as an Agent Orchestrator, enabling the development of graph-based applications that autonomously execute sequences of actions. In these applications, the Large Language Model (LLM) functions as the primary decision-maker, determining the subsequent steps. 

Following this, an AI service is created based on the previously constructed application.


## Table of Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Create AI service](#Create-AI-service)
3. [Testing AI service's function locally](#Testing-AI-service's-function-locally)
4. [Deploy AI service](#Deploy-AI-service)
5. [Example of executing an AI service](#Example-of-executing-an-AI-service)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install dependencies

**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U "langgraph>0.2,<0.3" | tail -n 1
%pip install -U "ibm_watsonx_ai>=1.3.6" | tail -n 1
%pip install -U "langchain_ibm>=0.3,<0.4" | tail -n 1

### Define the watsonx.ai credentials
Use the code cell below to define the watsonx.ai credentials that are required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">Managing user API keys</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

### Working with spaces

You need to create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=wx) to create one.

- Click **New Deployment Space**
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press **Create**
- Go to **Manage** tab
- Copy `Space GUID` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: assign space ID below

In [3]:
import os

try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

Create an instance of APIClient with authentication details.

In [4]:
from ibm_watsonx_ai import APIClient

api_client = APIClient(credentials=credentials, space_id=space_id)

Specify the `model_id` of the model you will use for the chat with tools.

In [5]:
model_id = "meta-llama/llama-3-3-70b-instruct"

<a id="Create-AI-service"></a>
## Create AI service

The content of this notebook is derived from and based on the material presented in the [Use watsonx, and `mistral-medium-2505` to make simple chat conversation and tool calls](https://github.com/IBM/watsonx-ai-samples/blob/master/cloud/notebooks/python_sdk/deployments/foundation_models/chat/Use%20watsonx%2C%20and%20%60mistral-medium-2505%60%20to%20make%20simple%20chat%20conversation%20and%20tool%20calls.ipynb) notebook. 

Prepare function which will be deployed using AI service.

In [6]:
def deployable_ai_service(context, url=credentials["url"], model_id=model_id):

    from ibm_watsonx_ai import APIClient, Credentials
    from langchain_core.tools import tool
    from langchain_ibm import ChatWatsonx
    from langgraph.prebuilt import create_react_agent

    api_client = APIClient(
        credentials=Credentials(url=url, token=context.generate_token()),
        space_id=context.get_space_id(),
    )

    chat = ChatWatsonx(
        watsonx_client=api_client, model_id=model_id, params={"temperature": 0.1}
    )

    @tool
    def add(a: float, b: float) -> float:
        """Add a and b."""
        return a + b

    @tool
    def subtract(a: float, b: float) -> float:
        """Subtract a and b."""
        return a - b

    @tool
    def multiply(a: float, b: float) -> float:
        """Multiply a and b."""
        return a * b

    @tool
    def divide(a: float, b: float) -> float:
        """Divide a and b."""
        return a / b

    tools = [add, subtract, multiply, divide]

    graph = create_react_agent(chat, tools=tools)

    def generate(context) -> dict:
        api_client.set_token(context.get_token())

        payload = context.get_json()
        question = payload["question"]

        response = graph.invoke({"messages": [("user", f"{question}")]})

        json_messages = [msg.to_json() for msg in response["messages"]]

        response["messages"] = json_messages

        return {"body": response}

    def generate_stream(context):
        api_client.set_token(context.get_token())

        payload = context.get_json()
        question = payload["question"]

        for el in graph.stream(
            {"messages": [("user", f"{question}")]}, stream_mode="values"
        ):
            json_messages = [msg.to_json() for msg in el["messages"]]
            el["messages"] = json_messages
            yield el

    return generate, generate_stream

Add a helpful function to print messages from the model.

In [7]:
import json


def print_message(message):
    last_message_id = message["id"][-1]
    print(f" ===== {last_message_id} =====")

    match last_message_id:
        case "AIMessage":
            content = message["kwargs"].get(
                "additional_kwargs", message["kwargs"].get("content")
            )
            print(
                json.dumps(content, indent=2) if isinstance(content, dict) else content,
                end="\n\n",
            )
        case "ToolMessage":
            print(message["kwargs"]["name"])
            print(message["kwargs"]["content"], end="\n\n")
        case _:
            print(message["kwargs"]["content"], end="\n\n")


def ai_services_pretty_print(iter):
    if "body" in iter:
        iter = iter["body"]

    for message in iter["messages"]:
        print_message(message)


def ai_services_pretty_print_stream(iter):
    for el in iter:
        try:
            el = json.loads(el)
        except TypeError:
            pass

        print_message(el["messages"][-1])

<a id="Testing-AI-service's-function-locally"></a>
## Testing AI service's function locally

You can test AI service's function locally. Initialise `RuntimeContext` firstly.

In [8]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(api_client=api_client)

In [9]:
local_function = deployable_ai_service(context=context)

Prepare request json payload.

In [10]:
context.request_payload_json = {
    "question": (
        "What is the total sum of the numbers 11, 13, and 20? "
        "Perform you calculations step by step. "
        "Remember to always return the final result using the last tool message."
    )
}

Execute the `generate` function locally.

In [11]:
resp = local_function[0](context)

In [12]:
ai_services_pretty_print(resp)

 ===== HumanMessage =====
What is the total sum of the numbers 11, 13, and 20? Perform you calculations step by step. Remember to always return the final result using the last tool message.

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "id": "chatcmpl-tool-45a8ed778e4547fa8d291b2e2b465580",
      "type": "function",
      "function": {
        "name": "add",
        "arguments": "{\"a\": \"11\", \"b\": \"13\"}"
      }
    }
  ]
}

 ===== ToolMessage =====
add
24.0

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "id": "chatcmpl-tool-6314b63c796a415399e92e555b284c1b",
      "type": "function",
      "function": {
        "name": "add",
        "arguments": "{\"a\": \"24\", \"b\": \"20\"}"
      }
    }
  ]
}

 ===== ToolMessage =====
add
44.0

 ===== AIMessage =====
The total sum of the numbers 11, 13, and 20 is 44.0.



Execute the `generate_stream` function locally.

In [13]:
response = local_function[1](context)

In [14]:
ai_services_pretty_print_stream(response)

 ===== HumanMessage =====
What is the total sum of the numbers 11, 13, and 20? Perform you calculations step by step. Remember to always return the final result using the last tool message.

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "id": "chatcmpl-tool-d576102534ea448aa7905c1efd75a823",
      "type": "function",
      "function": {
        "name": "add",
        "arguments": "{\"a\": \"11\", \"b\": \"13\"}"
      }
    }
  ]
}

 ===== ToolMessage =====
add
24.0

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "id": "chatcmpl-tool-e222da3a0a8a409c9493744d33b44acc",
      "type": "function",
      "function": {
        "name": "add",
        "arguments": "{\"a\": \"24\", \"b\": \"20\"}"
      }
    }
  ]
}

 ===== ToolMessage =====
add
44.0

 ===== AIMessage =====
The total sum of the numbers 11, 13, and 20 is 44.0.



<a id="Deploy-AI-service"></a>
## Deploy AI service

Store the AI service

In [15]:
sw_spec_id = api_client.software_specifications.get_id_by_name("genai-A25-py3.12")

meta_props = {
    api_client.repository.AIServiceMetaNames.NAME: "AI service SDK with langgraph",
    api_client.repository.AIServiceMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
}
stored_ai_service_details = api_client.repository.store_ai_service(
    deployable_ai_service, meta_props
)

Get AI service ID

In [16]:
ai_service_id = api_client.repository.get_ai_service_id(stored_ai_service_details)
ai_service_id

'99041ace-b951-4796-8d67-b7776c434856'

Create online deployment of AI service.

In [17]:
meta_props = {
    api_client.deployments.ConfigurationMetaNames.NAME: "AI service with tools",
    api_client.deployments.ConfigurationMetaNames.ONLINE: {},
}

deployment_details = api_client.deployments.create(ai_service_id, meta_props)



######################################################################################

Synchronous deployment creation for id: '99041ace-b951-4796-8d67-b7776c434856' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
...............
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='7943e6ac-ed1f-45d8-a9c1-5e3084112583'
-----------------------------------------------------------------------------------------------




Obtain the `deployment_id` of the previously created deployment.

In [18]:
deployment_id = api_client.deployments.get_id(deployment_details)

<a id="Example-of-executing-an-AI-service"></a>
## Example of executing an AI service

Execute `generate` method.

In [19]:
question = (
    "What is the total sum of the numbers 11, 13, and 20? "
    "Perform you calculations step by step. "
    "Remember to always return the final result using the last tool message."
)

deployments_results = api_client.deployments.run_ai_service(
    deployment_id, {"question": question}
)

In [20]:
ai_services_pretty_print(deployments_results)

 ===== HumanMessage =====
What is the total sum of the numbers 11, 13, and 20? Perform you calculations step by step. Remember to always return the final result using the last tool message.

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "function": {
        "arguments": "{\"a\": \"11\", \"b\": \"13\"}",
        "name": "add"
      },
      "id": "chatcmpl-tool-8245dc666101419e88405844160fe811",
      "type": "function"
    }
  ]
}

 ===== ToolMessage =====
add
24.0

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "function": {
        "arguments": "{\"a\": \"24\", \"b\": \"20\"}",
        "name": "add"
      },
      "id": "chatcmpl-tool-bcd11cc2949a4218a06637465d8668e4",
      "type": "function"
    }
  ]
}

 ===== ToolMessage =====
add
44.0

 ===== AIMessage =====
The total sum of the numbers 11, 13, and 20 is 44.0.



Execute `generate_stream` method.

In [21]:
question = (
    "Add 2 to 5. Result multiply by 3. Result divide by 10. "
    "Perform you calculations step by step. "
    "Always return the value from the last message to the user."
)

deployments_results = api_client.deployments.run_ai_service_stream(
    deployment_id, {"question": question}
)

In [22]:
ai_services_pretty_print_stream(deployments_results)

 ===== HumanMessage =====
Add 2 to 5. Result multiply by 3. Result divide by 10. Perform you calculations step by step. Always return the value from the last message to the user.

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "id": "chatcmpl-tool-6e2ae10c4b2f4108a597f3c2c44e5653",
      "type": "function",
      "function": {
        "name": "add",
        "arguments": "{\"a\": \"5\", \"b\": \"2\"}"
      }
    }
  ]
}

 ===== ToolMessage =====
add
7.0

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "id": "chatcmpl-tool-6610f8879a7f4688ac85bcd057e2461d",
      "type": "function",
      "function": {
        "name": "multiply",
        "arguments": "{\"a\": \"7.0\", \"b\": \"3\"}"
      }
    }
  ]
}

 ===== ToolMessage =====
multiply
21.0

 ===== AIMessage =====
{
  "tool_calls": [
    {
      "id": "chatcmpl-tool-2023726f5ad24d49a092fc33b596dd35",
      "type": "function",
      "function": {
        "name": "divide",
        "arguments": "{\"a\": \"21.0\", \"b\": \"1

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to create and deploy AI service using `ibm_watsonx_ai` SDK.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Author

**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2024-2026 IBM. This notebook and its source code are released under the terms of the MIT License.